# Document Question Answering System using RAG

This project implements a Retrieval-Augmented Generation (RAG) pipeline
for answering questions from custom PDF documents.

The system:
- extracts text from PDFs
- chunks text into smaller segments
- creates embeddings
- stores embeddings in a FAISS vector database
- retrieves relevant chunks
- generates grounded answers using a language model

# Install Required Libraries

This cell installs all dependencies required for the RAG system:
- LangChain
- FAISS
- Sentence Transformers
- PyPDF2
- Hugging Face Transformers
- Streamlit

In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
sentence-transformers \
faiss-cpu \
pypdf \
transformers \
accelerate

# Upload PDF Document

Upload your custom PDF document.
This document will be used for question answering.

In [3]:
from google.colab import files

uploaded = files.upload()

Saving Week7_Project.pdf to Week7_Project (2).pdf


# Extract Text from PDF

This step reads the uploaded PDF and extracts raw text data.

In [4]:
from pypdf import PdfReader

pdf_name = list(uploaded.keys())[0]

reader = PdfReader(pdf_name)

raw_text = ""

for page in reader.pages:
    raw_text += page.extract_text()

print(raw_text[:2000])

Document  Question  Answering  System  (RAG)  
Dataset:  
Simple  Beginner  Dataset  (Easiest)  
Use  your  own  PDFs:  
●  Notes  ●  Resume  ●  Research  papers  ●  Books  ●  RAG  is  meant  for  custom/private  data  
Or  Try  this  Hugging  Face  Dataset 
Reference:Github  Link   
Overview  
This  project  implements  a  Retrieval-Augmented  Generation  (RAG)  system  that  
answers
 
questions
 
based
 
on
 
custom
 
documents.
 
Instead  of  relying  only  on  a  language  model’s  internal  knowledge,  the  system  
retrieves
 
relevant
 
information
 
from
 
documents
 
and
 
then
 
generates
 
answers
 
grounded
 
in
 
that
 
information.
 
This
 
improves
 
factual
 
accuracy
 
and
 
allows
 
question
 
answering
 
over
 
private
 
or
 
domain-specific
 
data.
 
Objectives  
●  Understand  the  concept  of  Retrieval-Augmented  Generation  (RAG)  ●  Build  a  pipeline  combining  retrieval  and  generation  ●  Enable  question  answering  over  custom  documents  such  as  PDF

# Text Chunking

Large text is divided into smaller chunks for efficient retrieval.
Chunking improves semantic search accuracy.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_text(raw_text)

print("Total Chunks:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0])

Total Chunks: 11

First Chunk:

Document  Question  Answering  System  (RAG)  
Dataset:  
Simple  Beginner  Dataset  (Easiest)  
Use  your  own  PDFs:  
●  Notes  ●  Resume  ●  Research  papers  ●  Books  ●  RAG  is  meant  for  custom/private  data  
Or  Try  this  Hugging  Face  Dataset 
Reference:Github  Link   
Overview  
This  project  implements  a  Retrieval-Augmented  Generation  (RAG)  system  that  
answers
 
questions
 
based
 
on
 
custom
 
documents.


# Create Text Embeddings

Each chunk is converted into a numerical vector representation
using a Sentence Transformer embedding model.

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

embeddings = embedding_model.encode(chunks)

print("Embedding Shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Shape: (11, 384)


# Build Vector Database using FAISS

FAISS stores embeddings and enables fast similarity search.

In [8]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Total vectors stored:", index.ntotal)

Total vectors stored: 11


# Retrieval Function

This function retrieves the most relevant chunks
based on the user's question.

In [9]:
def retrieve_chunks(query, k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    retrieved = []

    for idx in indices[0]:
        retrieved.append(chunks[idx])

    return retrieved

# Test Similarity Search

This step validates whether relevant chunks
are being retrieved correctly.

In [10]:
query = "What is the main idea of the document?"

results = retrieve_chunks(query)

for i, chunk in enumerate(results):
    print(f"\n--- Retrieved Chunk {i+1} ---\n")
    print(chunk)


--- Retrieved Chunk 1 ---

questions
 
based
 
on
 
custom
 
documents.
 
Instead  of  relying  only  on  a  language  model’s  internal  knowledge,  the  system  
retrieves
 
relevant
 
information
 
from
 
documents
 
and
 
then
 
generates
 
answers
 
grounded
 
in
 
that
 
information.
 
This
 
improves
 
factual
 
accuracy
 
and
 
allows
 
question
 
answering
 
over
 
private
 
or
 
domain-specific
 
data.
 
Objectives

--- Retrieved Chunk 2 ---

Workflow  1.  Load  and  preprocess  documents  2.  Split  text  into  chunks  3.  Convert  chunks  into  embeddings  4.  Store  embeddings  in  a  vector  database  5.  Accept  user  query  6.  Retrieve  relevant  chunks  7.  Generate  answer  using  retrieved  context  
Example  Flow  
User  Question:  
“What
 
is
 
the
 
main
 
idea
 
of
 
the
 
document?”
 
System  Process:  
●  Retrieves  relevant  sections  ●  Provides  them  as  context  ●  Generates  a  concise  answer

--- Retrieved Chunk 3 ---

or
 
domain-specific
 
data.
 
O

# Load Language Model

FLAN-T5 is used for answer generation.
The model generates grounded responses using retrieved context.

In [13]:
from transformers import pipeline

llm = pipeline(
    task="text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

print("LLM Loaded Successfully")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', '

LLM Loaded Successfully


# RAG Answer Generation

The retrieved context and user question are combined
into a single prompt for answer generation.

In [14]:
def generate_answer(query):

    retrieved_docs = retrieve_chunks(query)

    context = "\n".join(retrieved_docs)

    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm(prompt)

    return response[0]['generated_text']

# Final End-to-End Testing

This is the complete Retrieval-Augmented Generation pipeline.

In [17]:
question = input("Enter your question: ")

answer = generate_answer(question)

print("\nGenerated Answer:\n")
print(answer)

Enter your question: Explain the workflow of the system.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:


    Answer the question using the context below.
    
    Context:
    Workflow  1.  Load  and  preprocess  documents  2.  Split  text  into  chunks  3.  Convert  chunks  into  embeddings  4.  Store  embeddings  in  a  vector  database  5.  Accept  user  query  6.  Retrieve  relevant  chunks  7.  Generate  answer  using  retrieved  context  
Example  Flow  
User  Question:  
“What
 
is
 
the
 
main
 
idea
 
of
 
the
 
document?”
 
System  Process:  
●  Retrieves  relevant  sections  ●  Provides  them  as  context  ●  Generates  a  concise  answer
from
 
a
 
document.
 
It
 
typically
 
uses
 
embeddings
 
and
 
vector
 
similarity
 
search.
 2.  Augmentation  
The
 
retrieved
 
content
 
is
 
added
 
to
 
the
 
model’s
 
input
 
to
 
provide
 
context
 
for
 
answering.
 3.  Generation  
A
 
language
 
model
 
generates
 
the
 
final
 
answer
 
using
 
the
 
retrieved
 
context,
 
ensuring
 
responses
 
are
 
grounded
 
in
 
actual
 
data.
 
System  Architecture  


# Validation Logs

This section validates retrieval performance
using dynamic sample queries.

In [18]:
  test_questions = [
    "What is the document about?",
    "What are the key concepts?",
    "Explain retrieval.",
    "Explain embeddings."
]

for q in test_questions:

    print("\n========================")
    print("Question:", q)

    retrieved = retrieve_chunks(q)

    print("\nRetrieved Context:")

    for i, r in enumerate(retrieved):
        print(f"\nChunk {i+1}:")
        print(r[:300])

    answer = generate_answer(q)

    print("\nGenerated Answer:")
    print(answer)


Question: What is the document about?

Retrieved Context:

Chunk 1:
questions
 
based
 
on
 
custom
 
documents.
 
Instead  of  relying  only  on  a  language  model’s  internal  knowledge,  the  system  
retrieves
 
relevant
 
information
 
from
 
documents
 
and
 
then
 
generates
 
answers
 
grounded
 
in
 
that
 
information.
 
This
 
improves
 
factual
 
accura

Chunk 2:
Workflow  1.  Load  and  preprocess  documents  2.  Split  text  into  chunks  3.  Convert  chunks  into  embeddings  4.  Store  embeddings  in  a  vector  database  5.  Accept  user  query  6.  Retrieve  relevant  chunks  7.  Generate  answer  using  retrieved  context  
Example  Flow  
User  Quest

Chunk 3:
from
 
a
 
document.
 
It
 
typically
 
uses
 
embeddings
 
and
 
vector
 
similarity
 
search.
 2.  Augmentation  
The
 
retrieved
 
content
 
is
 
added
 
to
 
the
 
model’s
 
input
 
to
 
provide
 
context
 
for
 
answering.
 3.  Generation  
A
 
language
 
model
 
generates
 
the
 
final
 
answe


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:

    Answer the question using the context below.
    
    Context:
    questions
 
based
 
on
 
custom
 
documents.
 
Instead  of  relying  only  on  a  language  model’s  internal  knowledge,  the  system  
retrieves
 
relevant
 
information
 
from
 
documents
 
and
 
then
 
generates
 
answers
 
grounded
 
in
 
that
 
information.
 
This
 
improves
 
factual
 
accuracy
 
and
 
allows
 
question
 
answering
 
over
 
private
 
or
 
domain-specific
 
data.
 
Objectives
Workflow  1.  Load  and  preprocess  documents  2.  Split  text  into  chunks  3.  Convert  chunks  into  embeddings  4.  Store  embeddings  in  a  vector  database  5.  Accept  user  query  6.  Retrieve  relevant  chunks  7.  Generate  answer  using  retrieved  context  
Example  Flow  
User  Question:  
“What
 
is
 
the
 
main
 
idea
 
of
 
the
 
document?”
 
System  Process:  
●  Retrieves  relevant  sections  ●  Provides  them  as  context  ●  Generates  a  concise  answer
from
 
a
 
document.
 
It

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:

    Answer the question using the context below.
    
    Context:
    These  are  typically  unstructured  and  may  contain  domain-specific  knowledge.  
Components  Used  
●  Embedding  model  for  converting  text  into  vectors  ●  Vector  store  for  similarity  search  ●  Language  model  for  generating  answers
or
 
domain-specific
 
data.
 
Objectives  
●  Understand  the  concept  of  Retrieval-Augmented  Generation  (RAG)  ●  Build  a  pipeline  combining  retrieval  and  generation  ●  Enable  question  answering  over  custom  documents  such  as  PDFs  or  text  files  ●  Learn  how  modern  AI  systems  work  internally  
Key  Concepts  
1.  Retrieval  
Retrieval
 
is
 
responsible
 
for
 
finding
 
the
 
most
 
relevant
 
chunks
 
of
 
text
 
from
 
a
 
document.
 
It
 
typically
 
uses
Key  Learnings  
●  How  RAG  systems  combine  retrieval  and  generation  ●  Importance  of  retrieval  in  improving  answer  accuracy  ●  Working  with  embeddi

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:

    Answer the question using the context below.
    
    Context:
    or
 
domain-specific
 
data.
 
Objectives  
●  Understand  the  concept  of  Retrieval-Augmented  Generation  (RAG)  ●  Build  a  pipeline  combining  retrieval  and  generation  ●  Enable  question  answering  over  custom  documents  such  as  PDFs  or  text  files  ●  Learn  how  modern  AI  systems  work  internally  
Key  Concepts  
1.  Retrieval  
Retrieval
 
is
 
responsible
 
for
 
finding
 
the
 
most
 
relevant
 
chunks
 
of
 
text
 
from
 
a
 
document.
 
It
 
typically
 
uses
database
 
for
 
efficient
 
similarity
 
search.
 5.  Query  Processing  
The
 
user’s
 
question
 
is
 
converted
 
into
 
an
 
embedding.
 6.  Context  Retrieval  
The
 
system
 
retrieves
 
the
 
most
 
relevant
 
chunks
 
from
 
the
 
database.
 7.  Answer  Generation  
A
 
language
 
model
 
generates
 
an
 
answer
 
using
 
the
 
retrieved
 
context.
 
Data  
Input  sources  include:  
●  PDF  documents  ●

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:

    Answer the question using the context below.
    
    Context:
    These  are  typically  unstructured  and  may  contain  domain-specific  knowledge.  
Components  Used  
●  Embedding  model  for  converting  text  into  vectors  ●  Vector  store  for  similarity  search  ●  Language  model  for  generating  answers
1.  Document  Ingestion  
Documents
 
such
 
as
 
PDFs
 
or
 
text
 
files
 
are
 
loaded
 
and
 
converted
 
into
 
raw
 
text.
 2.  Text  Chunking  
The
 
text
 
is
 
split
 
into
 
smaller
 
chunks
 
to
 
improve
 
retrieval
 
accuracy.
 3.  Embedding  Creation  
Each
 
chunk
 
is
 
converted
 
into
 
a
 
vector
 
representation
 
capturing
 
its
 
semantic
 
meaning.
 4.  Vector  Database  
Embeddings
 
are
 
stored
 
in
 
a
 
vector
 
database
 
for
 
efficient
 
similarity
 
search.
from
 
a
 
document.
 
It
 
typically
 
uses
 
embeddings
 
and
 
vector
 
similarity
 
search.
 2.  Augmentation  
The
 
retrieved
 
content
 
is
 
added
 
to
 
t

# Save FAISS Index

The vector database is saved for future reuse.

In [19]:
faiss.write_index(index, "vector_store.index")

print("Vector database saved successfully.")

Vector database saved successfully.
